# RootFetch Reproducibility Notebook v1

This notebook proves, from public immutable artifacts, that a run can be independently verified and recomputed using the published model contract implementation.

In [ ]:
import os
import sys
import json
import hashlib
from io import StringIO
from pathlib import Path

import requests
import pandas as pd

BASE_URL = os.getenv("ROOTFETCH_BASE_URL", "https://rootfetch.com").rstrip("/")
RUN_ID = (os.getenv("ROOTFETCH_RUN_ID") or "").strip() or None
TIMEOUT = float(os.getenv("ROOTFETCH_TIMEOUT", "30"))

repo_root = next((candidate for candidate in [Path.cwd(), *Path.cwd().parents] if (candidate / "rootfetch").exists()), None)
if repo_root is None:
    raise RuntimeError("Could not locate repo root containing /rootfetch package.")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from rootfetch.signals.model_v1 import compute_model_v1_from_growth, MODEL_VERSION, METHODOLOGY_VERSION

print({
    "base_url": BASE_URL,
    "requested_run_id": RUN_ID,
    "repo_root": str(repo_root),
    "model_version_impl": MODEL_VERSION,
    "methodology_version_impl": METHODOLOGY_VERSION,
})

In [ ]:
def artifact_url(relative_path: str) -> str:
    safe_path = relative_path.lstrip("/")
    return f"{BASE_URL}/rootfetch/artifacts/{safe_path}"

def fetch_bytes(relative_path: str) -> bytes:
    response = requests.get(artifact_url(relative_path), timeout=TIMEOUT)
    response.raise_for_status()
    return response.content

def fetch_json(relative_path: str):
    response = requests.get(artifact_url(relative_path), timeout=TIMEOUT)
    response.raise_for_status()
    return response.json()

if RUN_ID is None:
    latest_pointer = fetch_json("latest.json")
    RUN_ID = str(latest_pointer.get("run_id") or "").strip()
else:
    latest_pointer = None

if not RUN_ID:
    raise RuntimeError("Failed to resolve run_id.")

print({
    "resolved_run_id": RUN_ID,
    "run_url": f"{BASE_URL}/runs/{RUN_ID}",
})

In [ ]:
manifest = fetch_json(f"runs/{RUN_ID}/manifest.json")
files = manifest.get("files") if isinstance(manifest.get("files"), list) else []

verification_rows = []
for entry in files:
    path = str(entry.get("path") or "").strip()
    expected = str(entry.get("sha256") or "").strip().lower()
    payload = fetch_bytes(f"runs/{RUN_ID}/{path}")
    actual = hashlib.sha256(payload).hexdigest()
    verification_rows.append({
        "path": path,
        "expected_sha256": expected,
        "actual_sha256": actual,
        "match": expected == actual,
    })

verification_df = pd.DataFrame(verification_rows)
display(verification_df)

expected_count = len(verification_rows)
checked_count = int(verification_df["match"].count()) if len(verification_df.index) else 0
mismatch_count = int((~verification_df["match"]).sum()) if len(verification_df.index) else 0

if mismatch_count:
    raise AssertionError(f"Manifest verification failed with {mismatch_count} mismatches.")

print({
    "expected_count": expected_count,
    "checked_count": checked_count,
    "mismatch_count": mismatch_count,
    "status": "ok",
})

In [ ]:
model_artifact = fetch_json(f"runs/{RUN_ID}/model_latest.json")
coverage_artifact = fetch_json(f"runs/{RUN_ID}/coverage_latest.json")
signals_artifact = fetch_json(f"runs/{RUN_ID}/signals_latest.json")

print({
    "artifact_model_version": model_artifact.get("model_version"),
    "artifact_methodology_version": model_artifact.get("methodology_version"),
    "artifact_dvi": model_artifact.get("dvi"),
    "artifact_regime": model_artifact.get("regime"),
    "artifact_regime_confidence": model_artifact.get("regime_confidence"),
    "date_utc": model_artifact.get("date_utc"),
})

In [ ]:
growth_csv_text = requests.get(f"{BASE_URL}/rootfetch/growth_trends.csv", timeout=TIMEOUT).text
growth_df = pd.read_csv(StringIO(growth_csv_text))

growth_df["date_utc"] = growth_df.get("date_utc", "").astype(str)
growth_df["tld"] = growth_df.get("tld", "").astype(str)
growth_df["status"] = growth_df.get("status", "").astype(str).str.strip().str.lower()
growth_df["count_num"] = pd.to_numeric(growth_df.get("count"), errors="coerce")
growth_df["delta_abs_num"] = pd.to_numeric(growth_df.get("delta_abs"), errors="coerce")
growth_df["delta_pct_num"] = pd.to_numeric(growth_df.get("delta_pct"), errors="coerce")

target_date = str(model_artifact.get("effective_date_utc") or model_artifact.get("date_utc") or signals_artifact.get("date_utc") or "").strip()
approved_count = int(coverage_artifact.get("approved_tlds_count") or signals_artifact.get("approved_tlds_count") or 0)
if not target_date:
    raise RuntimeError("Could not resolve target date for model recomputation.")

recomputed = compute_model_v1_from_growth(
    growth_df=growth_df,
    date_utc=target_date,
    approved_tlds_count=approved_count,
)

print({
    "target_date": target_date,
    "approved_count": approved_count,
    "recomputed_dvi": recomputed.get("dvi"),
    "recomputed_regime": recomputed.get("regime"),
    "recomputed_regime_confidence": recomputed.get("regime_confidence"),
})

In [ ]:
artifact_dvi_raw = model_artifact.get("dvi")
if isinstance(artifact_dvi_raw, dict):
    artifact_dvi = float(artifact_dvi_raw.get("score") or 0.0)
else:
    artifact_dvi = float(artifact_dvi_raw or 0.0)
recomputed_dvi = float(recomputed.get("dvi") or 0.0)
dvi_diff = abs(recomputed_dvi - artifact_dvi)

artifact_regime = str(model_artifact.get("regime") or "")
recomputed_regime = str(recomputed.get("regime") or "")

artifact_conf = float(model_artifact.get("regime_confidence") or 0.0)
recomputed_conf = float(recomputed.get("regime_confidence") or 0.0)
conf_diff = abs(recomputed_conf - artifact_conf)

report = {
    "run_id": RUN_ID,
    "snapshot_ts_utc": manifest.get("snapshot_ts_utc"),
    "model_version": model_artifact.get("model_version"),
    "methodology_version": model_artifact.get("methodology_version"),
    "manifest": {
        "expected_count": expected_count,
        "checked_count": checked_count,
        "mismatch_count": mismatch_count,
    },
    "verification": {
        "dvi_match": dvi_diff <= 1e-9,
        "regime_match": recomputed_regime == artifact_regime,
        "confidence_match": conf_diff <= 1e-9,
        "model_version_match": str(recomputed.get("model_version")) == str(model_artifact.get("model_version")),
        "methodology_version_match": str(recomputed.get("methodology_version")) == str(model_artifact.get("methodology_version")),
    },
    "deltas": {
        "dvi_diff": dvi_diff,
        "confidence_diff": conf_diff,
    },
}

print(json.dumps(report, indent=2, sort_keys=True))

assert report["verification"]["dvi_match"], "DVI mismatch"
assert report["verification"]["regime_match"], "Regime mismatch"
assert report["verification"]["confidence_match"], "Regime confidence mismatch"
assert report["verification"]["model_version_match"], "Model version mismatch"
assert report["verification"]["methodology_version_match"], "Methodology version mismatch"

print("\n✅ Reproducibility checks passed")